In [1]:
import torch, torchvision
import mmdet
import mmcv
import mmocr

In [2]:
from mmcv import Config
cfg = Config.fromfile('/home/erik/Riksarkivet/Projects/fastighet/models/configs/satrn_plus.py')

In [3]:
from mmdet.apis import set_random_seed

# Set up working dir to save files and logs.
cfg.work_dir = '/home/erik/Riksarkivet/Projects/fastighet/models/satrn_htr_million_plus_test_height_resume1'

#cfg.load_from = '/home/erik/Riksarkivet/Projects/fastighet/models/satrn_htr_million_test_resume/epoch_2.pth'
#cfg.resume_from = '/home/erik/Riksarkivet/Projects/fastighet/models/satrn_htr_million/epoch_2.pth'

# The original learning rate (LR) is set for 8-GPU training.
# We divide it by 8 since we only use one GPU.
cfg.optimizer.lr = 0.001/8
cfg.lr_config.warmup = None
# Choose to log training results every 40 images to reduce the size of log file. 
cfg.log_config.interval = 500

# Set seed thus the results are more reproducible
cfg.seed = 0
set_random_seed(0, deterministic=False)
cfg.gpu_ids = range(1)
cfg.resume_from = '/home/erik/Riksarkivet/Projects/fastighet/models/satrn_htr_million_plus_test_height/epoch_2.pth'

# We can initialize the logger for training and have a look
# at the final config used for training
print(f'Config:\n{cfg.pretty_text}')

Config:
checkpoint_config = dict(interval=1)
log_config = dict(interval=500, hooks=[dict(type='TextLoggerHook')])
dist_params = dict(backend='nccl')
log_level = 'INFO'
load_from = None
resume_from = '/home/erik/Riksarkivet/Projects/fastighet/models/satrn_htr_million_plus_test_height/epoch_2.pth'
workflow = [('train', 1)]
label_convertor = dict(
    type='AttnConvertor', dict_type='DICT90', with_unknown=True, lower=True)
model = dict(
    type='SATRN',
    backbone=dict(type='ShallowCNN', input_channels=3, hidden_dim=512),
    encoder=dict(
        type='SatrnEncoder',
        n_layers=12,
        n_head=8,
        d_k=64,
        d_v=64,
        d_model=512,
        n_position=125,
        d_inner=2048,
        dropout=0.1),
    decoder=dict(
        type='NRTRDecoder',
        n_layers=6,
        d_embedding=512,
        n_head=8,
        d_model=512,
        d_inner=2048,
        d_k=64,
        d_v=64),
    loss=dict(type='TFLoss'),
    label_convertor=dict(
        type='AttnConver

In [4]:
from mmocr.datasets import build_dataset
from mmocr.models import build_detector
from mmocr.apis import train_detector
import os.path as osp

# Build dataset
datasets = [build_dataset(cfg.data.train)]

# Build the detector
model = build_detector(
    cfg.model, train_cfg=cfg.get('train_cfg'), test_cfg=cfg.get('test_cfg'))
# Add an attribute for visualization convenience
model.CLASSES = datasets[0].CLASSES

# Create work_dir
mmcv.mkdir_or_exist(osp.abspath(cfg.work_dir))
train_detector(model, datasets, cfg, distributed=False, validate=True)

/home/erik/Riksarkivet/Projects/fastighet/mmocr/mmocr/apis/train.py:82: UserWarning: config is now expected to have a `runner` section, please set `runner` in your config.
  'please set `runner` in your config.', UserWarning)
/home/erik/Riksarkivet/Projects/fastighet/mmocr/mmocr/apis/inference.py:88: UserWarning: Remove "MultiRotateAugOCR" to support batch inference since samples_per_gpu > 1.
  warnings.warn(warning_msg)
2022-04-12 08:04:26,389 - mmocr - INFO - load checkpoint from /home/erik/Riksarkivet/Projects/fastighet/models/satrn_htr_million_plus_test_height/epoch_2.pth
2022-04-12 08:04:26,391 - mmocr - INFO - Use load_from_local loader
2022-04-12 08:04:27,755 - mmocr - INFO - resumed epoch 2, iter 367266
2022-04-12 08:04:27,757 - mmocr - INFO - Start running, host: erik@erik-Precision-5820-Tower, work_dir: /home/erik/Riksarkivet/Projects/fastighet/models/satrn_htr_million_plus_test_height_resume1
2022-04-12 08:04:27,759 - mmocr - INFO - Hooks will be executed in the following or

In [5]:
from mmocr.apis import init_detector, model_inference
from mmocr.datasets.pipelines import MultiRotateAugOCR

img = '/home/erik/Riksarkivet/Projects/openmmlab/data/datasets/1930_census_prenames/B0001097/B0001097_00007/B0001097_00007_06.jpg'
checkpoint = '/home/erik/Riksarkivet/Projects/openmmlab/output/test_crnn/latest.pth'
#out_file = 'outputs/1036169.jpg'

model = init_detector(cfg, checkpoint, device="cuda:0")
if model.cfg.data.test['type'] == 'ConcatDataset':
    model.cfg.data.test.pipeline = model.cfg.data.test['datasets'][0].pipeline


result = model_inference(model, img)
print(f'result: {result}')

Use load_from_local loader
result: {'text': '', 'score': 0.0}
